In [16]:
from pathlib import Path
import pandas as pd



Cleaning the data

In [17]:
data_dir = Path("../dataSets")

train = pd.read_csv(data_dir / "train.csv", encoding="utf-8")
validation = pd.read_csv(data_dir / "validation.csv", encoding="utf-8")

for frame in (train, validation):
    frame["subject"] = frame["subject"].fillna("")
    frame["secondary_category"] = frame["secondary_category"].fillna("")

    # Checking for duplicate ticket IDs 
print("Duplicate train IDs:", train["ticket_id"].duplicated().sum())
print("Duplicate validation IDs:", validation["ticket_id"].duplicated().sum())
    

Duplicate train IDs: 0
Duplicate validation IDs: 0


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

def make_ticket_text(data):
    return (
        data["channel"].astype(str)
        + " [SUBJECT] " + data["subject"].fillna("").astype(str)
        + " [TICKET] " + data["text"].astype(str)
    )

train_text = make_ticket_text(train)
validation_text = make_ticket_text(validation)

category_model = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 5), min_df=2)),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

category_model.fit(train_text, train["category"])

predictions = category_model.predict(validation_text)
print(classification_report(validation["category"], predictions, zero_division=0))

from sklearn.metrics import accuracy_score

# Predict tickets the model used during training
train_predictions = category_model.predict(train_text)

# Predict tickets the model did not see during training
validation_predictions = category_model.predict(validation_text)

# Compare predictions with the correct categories
train_accuracy = accuracy_score(train["category"], train_predictions)
validation_accuracy = accuracy_score(
    validation["category"], validation_predictions
)

# Display both scores as percentages
print(f"Training accuracy:   {train_accuracy:.2%}")
print(f"Validation accuracy: {validation_accuracy:.2%}")

                     precision    recall  f1-score   support

      account_promo       0.54      0.58      0.56        72
      app_technical       0.55      0.25      0.34        64
     delivery_delay       0.49      0.53      0.51       120
       food_quality       0.36      0.38      0.37        56
    general_inquiry       0.52      0.78      0.62        40
          lost_item       0.33      0.47      0.39        40
order_missing_wrong       0.48      0.41      0.44        96
     payment_refund       0.47      0.45      0.46       144
    ride_trip_issue       0.80      0.73      0.76       112
     safety_conduct       0.57      0.75      0.65        32
    spam_irrelevant       1.00      1.00      1.00        24

           accuracy                           0.53       800
          macro avg       0.56      0.57      0.55       800
       weighted avg       0.54      0.53      0.53       800

Training accuracy:   99.33%
Validation accuracy: 53.25%


In [19]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

combined_model = Pipeline([
    # Read the same tickets in two ways, then join the resulting numbers
    ("features", FeatureUnion([

        # Recognize short character patterns, useful for spelling variations
        ("characters", TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 5),
            min_df=2
        )),

        # Recognize whole words and two-word phrases
        ("words", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            tokenizer=str.split,  # Split at spaces; preserve Sinhala/Tamil characters
            token_pattern=None    # Disable the default word-splitting rule
        ))
    ])),

    # Learn categories using both sets of text features
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

# Learn only from the training tickets
combined_model.fit(train_text, train["category"])

# Check familiar training tickets and unfamiliar validation tickets
combined_train_predictions = combined_model.predict(train_text)
combined_predictions = combined_model.predict(validation_text)

# Calculate the two accuracy scores
training_accuracy = accuracy_score(
    train["category"], combined_train_predictions
)
validation_accuracy = accuracy_score(
    validation["category"], combined_predictions
)

print(f"Training accuracy:   {training_accuracy:.2%}")
print(f"Validation accuracy: {validation_accuracy:.2%}")

# Show scores for each category, including macro average F1
print(classification_report(
    validation["category"],
    combined_predictions,
    zero_division=0
))

Training accuracy:   99.85%
Validation accuracy: 52.75%
                     precision    recall  f1-score   support

      account_promo       0.60      0.56      0.58        72
      app_technical       0.60      0.28      0.38        64
     delivery_delay       0.56      0.57      0.57       120
       food_quality       0.40      0.39      0.40        56
    general_inquiry       0.55      0.70      0.62        40
          lost_item       0.33      0.50      0.40        40
order_missing_wrong       0.41      0.36      0.39        96
     payment_refund       0.42      0.43      0.42       144
    ride_trip_issue       0.71      0.72      0.72       112
     safety_conduct       0.56      0.72      0.63        32
    spam_irrelevant       1.00      1.00      1.00        24

           accuracy                           0.53       800
          macro avg       0.56      0.57      0.55       800
       weighted avg       0.53      0.53      0.52       800



### Baseline model evaluation

The language results, misclassified-ticket examples, and confusion
matrix below use the original model: category_model.

Its validation accuracy was 53.25%.

The improved model, tuned_model, achieved 58.63%.
Its evaluation appears later in this notebook.

In [20]:
# Copy the language and actual category of each validation ticket
language_check = validation[["language", "category"]].copy()

# Get predictions from the trained model
language_check["prediction"] = category_model.predict(validation_text)

# True means correct; False means incorrect
language_check["correct"] = (
    language_check["category"] == language_check["prediction"]
)

# Count tickets and calculate accuracy for each language
summary = language_check.groupby("language").agg(
    tickets=("correct", "size"),
    accuracy=("correct", "mean")
)

# Convert accuracy from a decimal to a percentage
summary["accuracy_percent"] = (summary["accuracy"] * 100).round(2)

print(summary[["tickets", "accuracy_percent"]])

          tickets  accuracy_percent
language                           
en            280             46.07
mixed          20             55.00
si            160             40.62
singlish      200             72.00
ta            120             51.67
tanglish       20             75.00


In [21]:
results = validation[
    ["ticket_id", "channel", "subject", "text", "category"]
].copy()

results["predicted_category"] = predictions
errors = results[results["category"] != results["predicted_category"]]

print(f"Misclassified: {len(errors)} out of {len(validation)}")
print(errors.head(10).to_string(index=False))

Misclassified: 374 out of 800
   ticket_id         channel              subject                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     text            category  predicted_category
TF-VA-000432           email විස්තර පරීක්ෂා කරන්න                                                                                                                                                                                                                                      

In [22]:
from sklearn.metrics import confusion_matrix

labels = sorted(validation["category"].unique())
matrix = confusion_matrix(
    validation["category"],
    predictions,
    labels=labels
)

print(pd.DataFrame(matrix, index=labels, columns=labels))

                     account_promo  app_technical  delivery_delay  \
account_promo                   42              3               6   
app_technical                   17             16               7   
delivery_delay                   4              2              63   
food_quality                     0              0              17   
general_inquiry                  3              0               1   
lost_item                        0              2               7   
order_missing_wrong              6              3              12   
payment_refund                   4              2              11   
ride_trip_issue                  2              1               4   
safety_conduct                   0              0               1   
spam_irrelevant                  0              0               0   

                     food_quality  general_inquiry  lost_item  \
account_promo                   1                2          1   
app_technical                   5        

In [23]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, classification_report

# Make a fresh model with the same settings as the original
regularized_model = clone(category_model)

# Change C in the pipeline's "classifier" step
# A smaller C discourages the model from fitting training examples too closely
regularized_model.set_params(classifier__C=0.1)

# Train the new model
regularized_model.fit(train_text, train["category"])

# Get predictions for both datasets
regularized_train_predictions = regularized_model.predict(train_text)
regularized_predictions = regularized_model.predict(validation_text)

# Calculate accuracy
training_score = accuracy_score(
    train["category"], regularized_train_predictions
)
validation_score = accuracy_score(
    validation["category"], regularized_predictions
)

print(f"Training accuracy:   {training_score:.2%}")
print(f"Validation accuracy: {validation_score:.2%}")

# Show results for each category and the macro average F1
print(classification_report(
    validation["category"],
    regularized_predictions,
    zero_division=0
))

Training accuracy:   95.47%
Validation accuracy: 41.50%
                     precision    recall  f1-score   support

      account_promo       0.39      0.54      0.46        72
      app_technical       0.37      0.16      0.22        64
     delivery_delay       0.50      0.38      0.43       120
       food_quality       0.26      0.30      0.28        56
    general_inquiry       0.39      0.68      0.49        40
          lost_item       0.28      0.50      0.36        40
order_missing_wrong       0.42      0.34      0.38        96
     payment_refund       0.41      0.33      0.37       144
    ride_trip_issue       0.70      0.38      0.50       112
     safety_conduct       0.31      0.81      0.44        32
    spam_irrelevant       0.69      1.00      0.81        24

           accuracy                           0.41       800
          macro avg       0.43      0.49      0.43       800
       weighted avg       0.45      0.41      0.41       800



In [24]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

# Divide the training data into 3 parts, keeping similar category proportions.
# Each round learns from 2 parts and checks predictions on the remaining part.
folds = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Test four settings on a fresh copy of your original character model.
# Each fold learns its text features only from that fold's training tickets.
search = GridSearchCV(
    estimator=clone(category_model),
    param_grid={
        "classifier__C": [0.1, 1.0, 10.0, 100.0]
    },

    # Measure both accuracy and performance across all categories
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro"
    },

    # Choose the setting with the highest average macro F1,
    # then automatically train it using all the training tickets
    refit="macro_f1",
    cv=folds,
    n_jobs=1,    # Run one training job at a time to limit memory use
    verbose=2   # Show progress as each job finishes
)

# Search using training data only
search.fit(train_text, train["category"])

print("Best C:", search.best_params_["classifier__C"])
print(f"Best cross-validation macro F1: {search.best_score_:.3f}")

# Store the chosen model separately from your original model
tuned_model = search.best_estimator_

# Now check the chosen model on the validation tickets
tuned_predictions = tuned_model.predict(validation_text)

tuned_accuracy = accuracy_score(
    validation["category"], tuned_predictions
)
print(f"Validation accuracy: {tuned_accuracy:.2%}")

print(classification_report(
    validation["category"],
    tuned_predictions,
    digits=3,
    zero_division=0
))

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END ..................................classifier__C=0.1; total time=   3.9s
[CV] END ..................................classifier__C=0.1; total time=   4.2s
[CV] END ..................................classifier__C=0.1; total time=   3.6s
[CV] END ..................................classifier__C=1.0; total time=   6.6s
[CV] END ..................................classifier__C=1.0; total time=   6.9s
[CV] END ..................................classifier__C=1.0; total time=   7.7s
[CV] END .................................classifier__C=10.0; total time=  10.7s
[CV] END .................................classifier__C=10.0; total time=   9.6s
[CV] END .................................classifier__C=10.0; total time=   7.3s
[CV] END ................................classifier__C=100.0; total time=   9.0s
[CV] END ................................classifier__C=100.0; total time=   9.7s
[CV] END ................................classifi

In [25]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, classification_report

# Create a fresh model using the original character model's settings.
# This copies the settings, not its learned category predictions.
urgency_model = clone(category_model)

# Train using the urgency answers instead of the category answers.
# True = urgent; False = not urgent.
urgency_model.fit(train_text, train["is_urgent"])

# Predict urgency for the validation tickets
urgency_predictions = urgency_model.predict(validation_text)

# Compare predictions with the correct urgency answers
urgency_accuracy = accuracy_score(
    validation["is_urgent"],
    urgency_predictions
)
print(f"Urgency accuracy: {urgency_accuracy:.2%}")

# Show how well the model handles both urgent and nonurgent tickets
print(classification_report(
    validation["is_urgent"],
    urgency_predictions,
    labels=[False, True],
    target_names=["not urgent", "urgent"],
    zero_division=0
))

Urgency accuracy: 96.12%
              precision    recall  f1-score   support

  not urgent       0.98      0.98      0.98       720
      urgent       0.80      0.81      0.81        80

    accuracy                           0.96       800
   macro avg       0.89      0.90      0.89       800
weighted avg       0.96      0.96      0.96       800



In [26]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, classification_report

# Give empty secondary categories a name the model can learn.
# Later, we will convert "no_secondary" to None for the API.
no_secondary = "no_secondary"

secondary_train_labels = (
    train["secondary_category"].fillna("").replace("", no_secondary)
)
secondary_validation_labels = (
    validation["secondary_category"].fillna("").replace("", no_secondary)
)

# Create a fresh model using the original character model's settings
secondary_model = clone(category_model)

# Learn secondary categories using training tickets only
secondary_model.fit(train_text, secondary_train_labels)

# Predict secondary categories for validation tickets
secondary_predictions = secondary_model.predict(validation_text)

# Check overall accuracy
secondary_accuracy = accuracy_score(
    secondary_validation_labels,
    secondary_predictions
)
print(f"Secondary-category accuracy: {secondary_accuracy:.2%}")

# Check tickets that actually contain a second issue.
# A prediction of "no_secondary" counts as wrong for these tickets.
has_second_issue = secondary_validation_labels != no_secondary

second_issue_accuracy = accuracy_score(
    secondary_validation_labels[has_second_issue],
    secondary_predictions[has_second_issue.to_numpy()]
)
print(f"Accuracy on tickets with a second issue: {second_issue_accuracy:.2%}")

# Show performance for each secondary category
print(classification_report(
    secondary_validation_labels,
    secondary_predictions,
    zero_division=0
))

Secondary-category accuracy: 95.50%
Accuracy on tickets with a second issue: 56.25%
                     precision    recall  f1-score   support

      app_technical       1.00      0.44      0.62         9
     delivery_delay       1.00      0.44      0.62         9
       no_secondary       0.95      1.00      0.98       720
order_missing_wrong       1.00      0.33      0.50         9
     payment_refund       0.95      0.69      0.80        26
    ride_trip_issue       1.00      0.59      0.74        27

           accuracy                           0.95       800
          macro avg       0.98      0.58      0.71       800
       weighted avg       0.96      0.95      0.95       800



In [27]:
# Use this same version when saving the models
MODEL_VERSION = "neurox-v1"

# Required category-to-team mapping from the competition specification
TEAM_MAP = {
    "payment_refund": "Payments & Refunds",
    "ride_trip_issue": "Ride Operations",
    "lost_item": "Lost & Found",
    "order_missing_wrong": "Food Operations",
    "delivery_delay": "Delivery Operations",
    "food_quality": "Restaurant Quality",
    "account_promo": "Account Services",
    "safety_conduct": "Trust & Safety",
    "app_technical": "Tech Support",
    "general_inquiry": "Front-line Support",
    "spam_irrelevant": "Auto-close / Spam Filter"
}


def predict_ticket(channel, text, subject="", ticket_id=None):
    # Check that the inputs follow the required format
    if channel not in ("email", "chat", "call_transcript"):
        raise ValueError("Invalid channel.")

    if not isinstance(text, str) or not text.strip() or len(text) > 10000:
        raise ValueError("Text must be nonempty and at most 10,000 characters.")

    if not isinstance(subject, str) or len(subject) > 500:
        raise ValueError("Subject must be text, at most 500 characters.")

    if ticket_id is not None:
        if not isinstance(ticket_id, str) or len(ticket_id) > 64:
            raise ValueError("Ticket ID must be text, at most 64 characters.")

    # Put this ticket into a table with one row
    ticket = pd.DataFrame([{
        "channel": channel,
        "subject": subject,
        "text": text
    }])

    # Prepare the text exactly as we did during training
    ticket_text = make_ticket_text(ticket)

    # Get the main-category probabilities from our best category model
    probabilities = tuned_model.predict_proba(ticket_text)[0]

    # Choose the category with the highest probability
    best_index = probabilities.argmax()
    category = str(tuned_model.classes_[best_index])
    confidence = float(probabilities[best_index])

    # Ask the other two models for their predictions
    secondary = str(secondary_model.predict(ticket_text)[0])
    urgent = bool(urgency_model.predict(ticket_text)[0])

    # No second issue must become None (null when converted to JSON).
    # The secondary category must also differ from the main category.
    if secondary == "no_secondary" or secondary == category:
        secondary = None

    # Required rule: spam is nonurgent and has no secondary category
    if category == "spam_irrelevant":
        urgent = False
        secondary = None

    # Build the complete response
    result = {
        "category": category,
        "secondary_category": secondary,
        "team": TEAM_MAP[category],
        "is_urgent": urgent,
        "confidence": confidence,
        "model_version": MODEL_VERSION
    }

    # Include the ticket ID when the caller provided one
    if ticket_id is not None:
        result["ticket_id"] = ticket_id

    return result

In [28]:
result = predict_ticket(
    channel="chat",
    text="I was charged twice for the same ride.",
    ticket_id="demo-001"
)

print(result)

{'category': 'payment_refund', 'secondary_category': None, 'team': 'Payments & Refunds', 'is_urgent': False, 'confidence': 0.9973245711773366, 'model_version': 'neurox-v1', 'ticket_id': 'demo-001'}


In [29]:
import json
from jsonschema import Draft202012Validator
from sklearn.metrics import accuracy_score

# We will allow saving only after these checks finish successfully
validation_passed = False

# Load the competition's response rules
schema_path = (
    data_dir.parent
    / "appi nad schema"
    / "predict_response.schema.json"
)

with schema_path.open(encoding="utf-8") as file:
    response_schema = json.load(file)

validator = Draft202012Validator(response_schema)

# Run the combined function on every validation ticket
responses = []

for row in validation.to_dict(orient="records"):
    response = predict_ticket(
        channel=row["channel"],
        text=row["text"],
        subject=row["subject"],
        ticket_id=row["ticket_id"]
    )

    # Stop here if any response breaks the required rules
    validator.validate(response)

    responses.append(response)

# Turn the responses into a table so we can compare the answers
checked_results = pd.DataFrame(responses)

# Use empty strings to compare missing secondary categories consistently
actual_secondary = validation["secondary_category"].fillna("")
predicted_secondary = checked_results["secondary_category"].fillna("")

# Identify tickets that really contain a second issue
has_second_issue = actual_secondary.to_numpy() != ""

# Measure the FINAL predictions, after applying the response rules
metrics = {
    "main_category_accuracy": accuracy_score(
        validation["category"], checked_results["category"]
    ),
    "urgency_accuracy": accuracy_score(
        validation["is_urgent"], checked_results["is_urgent"]
    ),
    "secondary_accuracy": accuracy_score(
        actual_secondary, predicted_secondary
    ),
    "secondary_accuracy_when_present": accuracy_score(
        actual_secondary.to_numpy()[has_second_issue],
        predicted_secondary.to_numpy()[has_second_issue]
    )
}

validation_passed = True

print(f"Schema checks passed: {len(responses)}/{len(validation)}")

for name, score in metrics.items():
    print(f"{name}: {score:.2%}")

Schema checks passed: 800/800
main_category_accuracy: 58.63%
urgency_accuracy: 96.12%
secondary_accuracy: 95.50%
secondary_accuracy_when_present: 55.00%


In [30]:
import joblib
from importlib.metadata import version

assert validation_passed, "Run the validation cell successfully first."

# Create a folder beside your notebook
save_dir = Path("saved_models") / MODEL_VERSION
save_dir.mkdir(parents=True, exist_ok=True)

# Record package versions so your teammate can use the same environment
package_versions = {
    name: version(name)
    for name in ["scikit-learn", "pandas", "numpy", "scipy", "joblib"]
}

# Each model pipeline includes its trained TF-IDF text vectorizer
bundle = {
    "model_version": MODEL_VERSION,
    "category_model": tuned_model,
    "urgency_model": urgency_model,
    "secondary_model": secondary_model,
    "team_map": TEAM_MAP,
    "no_secondary_label": "no_secondary",
    "package_versions": package_versions
}

# Save all three models in one file
model_path = save_dir / "models.joblib"
joblib.dump(bundle, model_path, compress=3)

# Save the evaluation results separately for your report
with (save_dir / "validation_metrics.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metrics, file, indent=2)

# Record the required package versions
requirements = "\n".join(
    f"{name}=={number}"
    for name, number in package_versions.items()
)
(save_dir / "requirements.txt").write_text(
    requirements + "\n", encoding="utf-8"
)

# Reload the saved file
loaded = joblib.load(model_path)

# Confirm each saved model gives exactly the same class predictions
for name in ["category_model", "urgency_model", "secondary_model"]:
    original_predictions = bundle[name].predict(validation_text)
    loaded_predictions = loaded[name].predict(validation_text)

    assert (
        original_predictions == loaded_predictions
    ).all(), f"Predictions changed after loading {name}"

print("All three saved models loaded with matching predictions.")
print("Saved models:", model_path.resolve())

All three saved models loaded with matching predictions.
Saved models: C:\Users\Muqshith Akbar\Desktop\NeuroX\trained_model\saved_models\neurox-v1\models.joblib
